# Signal Scanning

This notebook explores the **Signal Scanning layer** of `equity_lake`:

- **`SignalScanner`** — orchestrator that runs all enabled generators across a watchlist
- **`SignalConfig`** — configuration loaded from `config/signals.yaml` controlling backtest, sentiment, and ML generators
- **`Watchlist`** — ticker list loaded from `config/watchlist.yaml` with optional groupings
- **Generators** — `BacktestSignalGenerator`, `SentimentSignalGenerator`, `MLPredictionSignalGenerator`, `MetaLabelSignalGenerator`
- **Formatters** — JSON, Markdown, and terminal table output
- **History** — Parquet-backed signal persistence

**Prerequisites:** Ingested price data in `data/lake/us_equity/`, news data in `data/lake/us_news/`, and optionally trained ML models.

In [1]:
import sys
sys.path.insert(0, "../src")

from equity_lake.signals import Signal, SignalConfig, Watchlist
from equity_lake.signals.config import load_signal_config, load_watchlist
from equity_lake.signals.scanner import SignalScanner
from equity_lake.signals.history import load_signals, save_signals, SIGNALS_DIR
from equity_lake.signals.generators import (
    SignalGenerator,
    BacktestSignalGenerator,
    SentimentSignalGenerator,
    MLPredictionSignalGenerator,
)
from equity_lake.signals.generators.meta_label import MetaLabelSignalGenerator
from equity_lake.signals.models import Signal as SignalModel
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd
import numpy as np
from datetime import date, timedelta
from pathlib import Path
import json

print("Signal scanning modules loaded")

Signal scanning modules loaded


## 1. Signal Configuration & Watchlist Setup

Load signal generator configuration from `config/signals.yaml` and the watchlist from `config/watchlist.yaml`. Each generator (backtest, sentiment, ML) has its own settings and an `enabled` flag.

In [2]:
try:
    signal_config = load_signal_config()
    print("Signal config loaded from config/signals.yaml")
    print(f"  backtest enabled: {signal_config.is_generator_enabled('backtest')}")
    print(f"  sentiment enabled: {signal_config.is_generator_enabled('sentiment')}")
    print(f"  ml enabled: {signal_config.is_generator_enabled('ml')}")
    print(f"  aggregation: {signal_config.aggregation}")
except Exception as e:
    print(f"Config load error: {e}")
    print("\nFalling back to inline SignalConfig...")
    signal_config = SignalConfig(
        backtest={"enabled": True, "strategies": [{"name": "momentum", "lookback_days": 20, "buy_threshold": 0.02, "sell_threshold": -0.01}], "min_win_rate": 0.55},
        sentiment={"enabled": True, "buy_threshold": 0.5, "sell_threshold": -0.3, "min_articles": 3, "lookback_days": 7},
        ml={"enabled": True, "model_dir": "data/models", "mode": "v1_direction", "horizon_days": 5, "min_confidence": 60},
        aggregation={"agreement_boost": 10, "unanimous_boost": 20},
    )

Config load error: Signal config not found: config/signals.yaml

Falling back to inline SignalConfig...


In [3]:
try:
    watchlist = load_watchlist()
    print(f"Watchlist: {watchlist.name}")
    print(f"  Description: {watchlist.description}")
    print(f"  Tickers: {watchlist.tickers}")
    if watchlist.groups:
        print(f"  Groups: {list(watchlist.groups.keys())}")
except Exception as e:
    print(f"Watchlist load error: {e}")
    print("\nFalling back to inline Watchlist...")
    watchlist = Watchlist(
        name="Demo Portfolio",
        description="Fallback watchlist for demo",
        tickers=["AAPL", "GOOGL", "MSFT", "TSLA", "NVDA"],
        groups={"tech": ["AAPL", "GOOGL", "MSFT", "NVDA"], "ev": ["TSLA"]},
    )
    print(f"Watchlist: {watchlist.name}")
    print(f"  Tickers: {watchlist.tickers}")

Watchlist load error: Watchlist config not found: config/watchlist.yaml

Falling back to inline Watchlist...
Watchlist: Demo Portfolio
  Tickers: ['AAPL', 'GOOGL', 'MSFT', 'TSLA', 'NVDA']


## 2. Create SignalScanner & Run All Generators

`SignalScanner` initializes all enabled generators from the config. Calling `scan(target_date)` runs each generator for every ticker in the watchlist and returns a flat list of `Signal` objects.

In [4]:
scanner = SignalScanner(config=signal_config, watchlist=watchlist)
print(f"Scanner initialized with {len(scanner.generators)} generator(s):")
for gen in scanner.generators:
    print(f"  - {gen.__class__.__name__} (enabled={gen.is_enabled()})")

target_date = date.today() - timedelta(days=1)
print(f"\nScanning for target_date={target_date}...")

try:
    signals = scanner.scan(target_date)
    print(f"Generated {len(signals)} signal(s)")
except Exception as e:
    print(f"Scan error: {e}")
    signals = []

2026-06-09 09:26:15 [info     ] duckdb_feature_view_ready      model_mode=v1_direction


Scanner initialized with 3 generator(s):
  - BacktestSignalGenerator (enabled=True)
  - SentimentSignalGenerator (enabled=True)
  - MLPredictionSignalGenerator (enabled=True)

Scanning for target_date=2026-06-08...


Generated 0 signal(s)


In [5]:
if signals:
    rows = []
    for s in signals:
        rows.append({
            "ticker": s.ticker,
            "date": s.date,
            "signal_type": s.signal_type,
            "action": s.action,
            "confidence": s.confidence,
            "reasoning": s.reasoning,
        })
    signals_df = pd.DataFrame(rows)
    print(f"Signals DataFrame shape: {signals_df.shape}")
    signals_df
else:
    print("No signals generated. Creating synthetic signals for visualization demo...")
    np.random.seed(42)
    demo_tickers = ["AAPL", "GOOGL", "MSFT", "TSLA", "NVDA"]
    demo_types = ["backtest", "sentiment", "ml"]
    demo_actions = ["BUY", "SELL"]
    rows = []
    for ticker in demo_tickers:
        for stype in np.random.choice(demo_types, size=np.random.randint(1, 4), replace=False):
            action = np.random.choice(demo_actions)
            conf = round(np.random.uniform(55, 95), 1)
            rows.append({
                "ticker": ticker,
                "date": target_date,
                "signal_type": stype,
                "action": action,
                "confidence": conf,
                "reasoning": f"{stype.title()} signal: {action} {ticker} ({conf:.0f}% confidence)",
            })
    signals_df = pd.DataFrame(rows)

    from equity_lake.signals.models import Signal as Sig
    signals = []
    for _, r in signals_df.iterrows():
        signals.append(Sig(
            ticker=r["ticker"], date=r["date"], signal_type=r["signal_type"],
            action=r["action"], confidence=r["confidence"],
            reasoning=r["reasoning"], metadata={},
        ))
    print(f"Demo signals: {len(signals)}")
    signals_df

No signals generated. Creating synthetic signals for visualization demo...
Demo signals: 11


## 3. Signal Timeline Visualization

Plot BUY/SELL signal markers on a price chart. Green triangle-up for BUY, red triangle-down for SELL. The background shows historical closing prices for context.

In [6]:
ticker_to_plot = "AAPL"

try:
    import duckdb
    con = duckdb.connect(":memory:")
    price_query = f"""
    SELECT date, ticker, close
    FROM read_parquet('data/lake/us_equity/date=*/*.parquet', hive_partitioning=1)
    WHERE ticker = '{ticker_to_plot}'
    ORDER BY date DESC
    LIMIT 120
    """
    price_df = con.execute(price_query).df()
    con.close()

    if price_df.empty:
        raise ValueError("No price data")
    price_df = price_df.sort_values("date")

    ticker_signals = [s for s in signals if s.ticker == ticker_to_plot]
    signal_dates = [s.date for s in ticker_signals]
    signal_actions = [s.action for s in ticker_signals]
    signal_confidences = [s.confidence for s in ticker_signals]

    buy_mask = [a == "BUY" for a in signal_actions]
    sell_mask = [a == "SELL" for a in signal_actions]

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=price_df["date"], y=price_df["close"],
        mode="lines", name=f"{ticker_to_plot} Close",
        line=dict(color="#636EFA", width=2),
    ))

    if any(buy_mask):
        buy_dates = [d for d, m in zip(signal_dates, buy_mask) if m]
        buy_prices = []
        for d in buy_dates:
            match = price_df[price_df["date"] == d]
            buy_prices.append(match["close"].iloc[0] if not match.empty else price_df["close"].iloc[-1])
        buy_confs = [c for c, m in zip(signal_confidences, buy_mask) if m]
        fig.add_trace(go.Scatter(
            x=buy_dates, y=buy_prices,
            mode="markers", name="BUY Signal",
            marker=dict(symbol="triangle-up", size=14, color="#00CC96"),
            hovertext=[f"BUY {c:.0f}%" for c in buy_confs],
        ))

    if any(sell_mask):
        sell_dates = [d for d, m in zip(signal_dates, sell_mask) if m]
        sell_prices = []
        for d in sell_dates:
            match = price_df[price_df["date"] == d]
            sell_prices.append(match["close"].iloc[0] if not match.empty else price_df["close"].iloc[-1])
        sell_confs = [c for c, m in zip(signal_confidences, sell_mask) if m]
        fig.add_trace(go.Scatter(
            x=sell_dates, y=sell_prices,
            mode="markers", name="SELL Signal",
            marker=dict(symbol="triangle-down", size=14, color="#EF553B"),
            hovertext=[f"SELL {c:.0f}%" for c in sell_confs],
        ))

    fig.update_layout(
        title=f"Signal Timeline — {ticker_to_plot}",
        xaxis_title="Date", yaxis_title="Price",
        height=500,
    )
    fig.show()
except Exception as e:
    print(f"Price chart error: {e}")
    print("\nGenerating synthetic price + signal chart for demo...")
    np.random.seed(42)
    dates = pd.date_range(end=target_date, periods=120, freq="B")
    prices = 150 + np.cumsum(np.random.normal(0.1, 1.5, len(dates)))

    demo_buy_idx = np.random.choice(len(dates), size=4, replace=False)
    demo_sell_idx = np.random.choice([i for i in range(len(dates)) if i not in demo_buy_idx], size=3, replace=False)

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=dates, y=prices, mode="lines", name="Price", line=dict(color="#636EFA", width=2)))
    fig.add_trace(go.Scatter(
        x=dates[demo_buy_idx], y=prices[demo_buy_idx],
        mode="markers", name="BUY",
        marker=dict(symbol="triangle-up", size=14, color="#00CC96"),
    ))
    fig.add_trace(go.Scatter(
        x=dates[demo_sell_idx], y=prices[demo_sell_idx],
        mode="markers", name="SELL",
        marker=dict(symbol="triangle-down", size=14, color="#EF553B"),
    ))
    fig.update_layout(
        title=f"Signal Timeline — {ticker_to_plot} (Synthetic Demo)",
        xaxis_title="Date", yaxis_title="Price", height=500,
    )
    fig.show()

Price chart error: IO Error: No files found that match the pattern "data/lake/us_equity/date=*/*.parquet"

LINE 3:     FROM read_parquet('data/lake/us_equity/date=*/*.parquet', hive_p...
                 ^

Generating synthetic price + signal chart for demo...


## 4. Confidence Distribution by Generator

Violin plot showing how confidence scores are distributed across each `signal_type` (backtest, sentiment, ml). This helps assess which generators tend to produce higher-conviction signals.

In [7]:
try:
    if not signals_df.empty:
        fig = px.violin(
            signals_df, x="signal_type", y="confidence",
            color="signal_type", box=True, points="all",
            title="Confidence Distribution by Generator Type",
            labels={"signal_type": "Generator Type", "confidence": "Confidence Score (0-100)"},
            color_discrete_map={
                "backtest": "#636EFA",
                "sentiment": "#FFA15A",
                "ml": "#00CC96",
            },
        )
        fig.update_layout(height=500, showlegend=False)
        fig.add_hline(y=60, line_dash="dash", line_color="gray",
                       annotation_text="Min confidence threshold")
        fig.show()
    else:
        raise ValueError("No signals data")
except Exception as e:
    print(f"Violin plot error: {e}")
    print("\nGenerating synthetic confidence distribution for demo...")
    np.random.seed(42)
    synth_violin = pd.DataFrame({
        "signal_type": np.random.choice(["backtest", "sentiment", "ml"], size=60, p=[0.4, 0.3, 0.3]),
        "confidence": np.concatenate([
            np.random.normal(70, 10, 24),
            np.random.normal(62, 12, 18),
            np.random.normal(75, 8, 18),
        ]),
    })
    synth_violin["confidence"] = synth_violin["confidence"].clip(30, 100)
    fig = px.violin(
        synth_violin, x="signal_type", y="confidence",
        color="signal_type", box=True, points="all",
        title="Confidence Distribution by Generator Type (Synthetic Demo)",
        labels={"signal_type": "Generator Type", "confidence": "Confidence Score (0-100)"},
        color_discrete_map={"backtest": "#636EFA", "sentiment": "#FFA15A", "ml": "#00CC96"},
    )
    fig.update_layout(height=500, showlegend=False)
    fig.add_hline(y=60, line_dash="dash", line_color="gray", annotation_text="Min confidence threshold")
    fig.show()

## 5. Multi-Generator Agreement Heatmap

For each ticker, check how many generators agree on the action. A heatmap shows:
- **Rows**: tickers
- **Columns**: generator types
- **Cell values**: +1 for BUY, -1 for SELL, 0 for no signal

This identifies tickers where multiple generators converge on the same direction.

In [8]:
try:
    if not signals_df.empty:
        action_map = {"BUY": 1, "SELL": -1, "HOLD": 0}
        heatmap_data = signals_df.pivot_table(
            index="ticker",
            columns="signal_type",
            values="action",
            aggfunc=lambda x: action_map.get(x.iloc[0], 0) if len(x) > 0 else 0,
        ).fillna(0).astype(int)

        fig = go.Figure(data=go.Heatmap(
            z=heatmap_data.values,
            x=heatmap_data.columns.tolist(),
            y=heatmap_data.index.tolist(),
            text=heatmap_data.values,
            texttemplate="%{text}",
            textfont={"size": 14},
            colorscale=[[0, "#EF553B"], [0.5, "#F5F5F5"], [1, "#00CC96"]],
            zmin=-1, zmax=1,
        ))
        fig.update_layout(
            title="Multi-Generator Agreement Heatmap",
            xaxis_title="Generator", yaxis_title="Ticker",
            height=max(400, 80 * len(heatmap_data)),
        )
        fig.show()

        agree_count = (heatmap_data.sum(axis=1) == heatmap_data.shape[1]).sum()
        disagree_count = (heatmap_data.abs().sum(axis=1) > 1).sum() - agree_count
        print(f"Tickers with full agreement: {agree_count}")
        print(f"Tickers with disagreement: {disagree_count}")
    else:
        raise ValueError("No signals data")
except Exception as e:
    print(f"Heatmap error: {e}")
    print("\nGenerating synthetic agreement heatmap for demo...")
    np.random.seed(42)
    demo_tickers = ["AAPL", "GOOGL", "MSFT", "TSLA", "NVDA", "AMZN", "META"]
    demo_gens = ["backtest", "sentiment", "ml"]
    synth_heatmap = pd.DataFrame(
        np.random.choice([-1, 0, 1], size=(len(demo_tickers), len(demo_gens)), p=[0.25, 0.2, 0.55]),
        index=demo_tickers, columns=demo_gens,
    )
    fig = go.Figure(data=go.Heatmap(
        z=synth_heatmap.values,
        x=synth_heatmap.columns.tolist(),
        y=synth_heatmap.index.tolist(),
        text=synth_heatmap.values,
        texttemplate="%{text}",
        textfont={"size": 14},
        colorscale=[[0, "#EF553B"], [0.5, "#F5F5F5"], [1, "#00CC96"]],
        zmin=-1, zmax=1,
    ))
    fig.update_layout(
        title="Multi-Generator Agreement Heatmap (Synthetic Demo)",
        xaxis_title="Generator", yaxis_title="Ticker",
        height=400,
    )
    fig.show()

Tickers with full agreement: 1
Tickers with disagreement: 3


## 6. Format Signals — JSON, Markdown, Terminal Table

`SignalScanner.format_signals()` delegates to pluggable formatters. Three built-in formats:
- **`json`**: machine-readable JSON array
- **`md`**: Markdown report with summary tables
- **`table`**: terminal-friendly formatted table

In [9]:
if signals:
    print("=" * 60)
    print("FORMAT: JSON")
    print("=" * 60)
    try:
        json_output = scanner.format_signals(signals, format="json")
        parsed = json.loads(json_output)
        print(json.dumps(parsed[:2], indent=2))
        if len(parsed) > 2:
            print(f"... ({len(parsed) - 2} more signals)")
    except Exception as e:
        print(f"JSON format error: {e}")
else:
    print("No signals to format")

FORMAT: JSON
[
  {
    "ticker": "AAPL",
    "date": "2026-06-08",
    "signal_type": "sentiment",
    "action": "BUY",
    "confidence": 86.2,
    "reasoning": "Sentiment signal: BUY AAPL (86% confidence)",
    "metadata": {}
  },
  {
    "ticker": "AAPL",
    "date": "2026-06-08",
    "signal_type": "ml",
    "action": "BUY",
    "confidence": 61.2,
    "reasoning": "Ml signal: BUY AAPL (61% confidence)",
    "metadata": {}
  }
]
... (9 more signals)


In [10]:
if signals:
    print("=" * 60)
    print("FORMAT: Markdown")
    print("=" * 60)
    try:
        md_output = scanner.format_signals(signals, format="md")
        print(md_output[:1500])
        if len(md_output) > 1500:
            print(f"... (truncated, {len(md_output)} total chars)")
    except Exception as e:
        print(f"Markdown format error: {e}")
else:
    print("No signals to format")

FORMAT: Markdown
# Signal Report
**Generated:** 2026-06-08  
**Total Signals:** 11

## Summary by Action

| Action | Count ||--------|-------|| BUY | 5 || SELL | 6 || HOLD | 0 |
## Backtest Signals

| Ticker | Action | Confidence | Reasoning ||--------|--------|------------|-----------|| AAPL | BUY | 59 | Backtest signal: BUY AAPL (59% confidence) || GOOGL | SELL | 88 | Backtest signal: SELL GOOGL (88% confidence) || MSFT | SELL | 62 | Backtest signal: SELL MSFT (62% confidence) || NVDA | SELL | 86 | Backtest signal: SELL NVDA (86% confidence) |
## Sentiment Signals

| Ticker | Action | Confidence | Reasoning ||--------|--------|------------|-----------|| AAPL | BUY | 86 | Sentiment signal: BUY AAPL (86% confidence) || GOOGL | SELL | 57 | Sentiment signal: SELL GOOGL (57% confidence) || TSLA | BUY | 80 | Sentiment signal: BUY TSLA (80% confidence) || NVDA | BUY | 70 | Sentiment signal: BUY NVDA (70% confidence) |
## Ml Signals

| Ticker | Action | Confidence | Reasoning ||--------|----

In [11]:
if signals:
    print("=" * 60)
    print("FORMAT: Terminal Table")
    print("=" * 60)
    try:
        table_output = scanner.format_signals(signals, format="table")
        print(table_output)
    except Exception as e:
        print(f"Table format error: {e}")
else:
    print("No signals to format")

FORMAT: Terminal Table
SIGNAL REPORT - 2026-06-08
Total Signals: 11

SUMMARY:
  BUY: 5
  SELL: 6
  HOLD: 0


BACKTEST SIGNALS:
--------------------------------------------------------------------------------
Ticker    Action      Conf  Reasoning
--------  --------  ------  --------------------------------------------
AAPL      BUY           59  Backtest signal: BUY AAPL (59% confidence)
GOOGL     SELL          88  Backtest signal: SELL GOOGL (88% confidence)
MSFT      SELL          62  Backtest signal: SELL MSFT (62% confidence)
NVDA      SELL          86  Backtest signal: SELL NVDA (86% confidence)


SENTIMENT SIGNALS:
--------------------------------------------------------------------------------
Ticker    Action      Conf  Reasoning
--------  --------  ------  ---------------------------------------------
AAPL      BUY           86  Sentiment signal: BUY AAPL (86% confidence)
GOOGL     SELL          57  Sentiment signal: SELL GOOGL (57% confidence)
TSLA      BUY           80  Senti

## 7. Signal History — Parquet Persistence

Signals are saved to partitioned Parquet files under `data/signals/date=YYYY-MM-DD/`. Use `save_signals()` to persist and `load_signals()` to retrieve historical signals.

In [12]:
try:
    if signals:
        save_signals(signals, target_date)
        print(f"Saved {len(signals)} signals to {SIGNALS_DIR}/date={target_date}/signals.parquet")
    else:
        print("No signals to save")
except Exception as e:
    print(f"Save error: {e}")

try:
    loaded = load_signals(target_date)
    if loaded:
        print(f"\nLoaded {len(loaded)} historical signals for {target_date}")
        for s in loaded[:5]:
            print(f"  {s.ticker:6s} | {s.action:4s} | {s.signal_type:10s} | conf={s.confidence:.0f}% | {s.reasoning[:60]}")
        if len(loaded) > 5:
            print(f"  ... ({len(loaded) - 5} more)")
    else:
        print(f"\nNo historical signals found for {target_date}")
except Exception as e:
    print(f"Load error: {e}")

Saved 11 signals to data/signals/date=2026-06-08/signals.parquet

Loaded 11 historical signals for 2026-06-08
  AAPL   | BUY  | sentiment  | conf=86% | Sentiment signal: BUY AAPL (86% confidence)
  AAPL   | BUY  | ml         | conf=61% | Ml signal: BUY AAPL (61% confidence)
  AAPL   | BUY  | backtest   | conf=59% | Backtest signal: BUY AAPL (59% confidence)
  GOOGL  | SELL | ml         | conf=83% | Ml signal: SELL GOOGL (83% confidence)
  GOOGL  | SELL | sentiment  | conf=57% | Sentiment signal: SELL GOOGL (57% confidence)
  ... (6 more)


In [13]:
try:
    signals_dir = Path("data/signals")
    if signals_dir.exists():
        partitions = sorted(signals_dir.glob("date=*"))
        print(f"Signal history partitions: {len(partitions)}")
        for p in partitions[-10:]:
            date_str = p.name.split("=")[1]
            try:
                df = pd.read_parquet(p / "signals.parquet")
                print(f"  {date_str}: {len(df)} signals — tickers: {df['ticker'].unique().tolist()[:5]}")
            except Exception:
                print(f"  {date_str}: (read error)")
    else:
        print("No signal history directory found")
        print("Run `equity signal scan` to generate signals")
except Exception as e:
    print(f"History scan error: {e}")

Signal history partitions: 1
  2026-06-08: 11 signals — tickers: ['AAPL', 'GOOGL', 'MSFT', 'TSLA', 'NVDA']


## 8. Signal Summary Dashboard

Combined overview: signal count by action and generator type, plus top-conviction signals.

In [14]:
try:
    if not signals_df.empty:
        from plotly.subplots import make_subplots

        fig = make_subplots(
            rows=1, cols=2,
            subplot_titles=("Signals by Action", "Signals by Generator"),
        )

        action_counts = signals_df["action"].value_counts()
        fig.add_trace(go.Bar(
            x=action_counts.index, y=action_counts.values,
            marker_color=["#00CC96" if a == "BUY" else "#EF553B" if a == "SELL" else "#636EFA" for a in action_counts.index],
            name="By Action",
        ), row=1, col=1)

        type_counts = signals_df["signal_type"].value_counts()
        fig.add_trace(go.Bar(
            x=type_counts.index, y=type_counts.values,
            marker_color=["#636EFA", "#FFA15A", "#00CC96"][:len(type_counts)],
            name="By Generator",
        ), row=1, col=2)

        fig.update_layout(
            title=f"Signal Summary — {target_date}",
            height=400, showlegend=False,
        )
        fig.show()

        top_signals = signals_df.nlargest(10, "confidence")[["ticker", "signal_type", "action", "confidence", "reasoning"]]
        print("\nTop 10 Highest-Confidence Signals:")
        print(top_signals.to_string(index=False))
    else:
        raise ValueError("No signals data")
except Exception as e:
    print(f"Summary error: {e}")


Top 10 Highest-Confidence Signals:
ticker signal_type action  confidence                                    reasoning
 GOOGL    backtest   SELL        88.3 Backtest signal: SELL GOOGL (88% confidence)
  NVDA    backtest   SELL        86.4  Backtest signal: SELL NVDA (86% confidence)
  AAPL   sentiment    BUY        86.2  Sentiment signal: BUY AAPL (86% confidence)
 GOOGL          ml   SELL        83.3       Ml signal: SELL GOOGL (83% confidence)
  MSFT          ml   SELL        79.5        Ml signal: SELL MSFT (80% confidence)
  TSLA   sentiment    BUY        79.5  Sentiment signal: BUY TSLA (80% confidence)
  NVDA   sentiment    BUY        70.3  Sentiment signal: BUY NVDA (70% confidence)
  MSFT    backtest   SELL        62.3  Backtest signal: SELL MSFT (62% confidence)
  AAPL          ml    BUY        61.2         Ml signal: BUY AAPL (61% confidence)
  AAPL    backtest    BUY        59.0   Backtest signal: BUY AAPL (59% confidence)


## Next Steps

- **[08-backtesting.ipynb](08-backtesting.ipynb)** — Run full backtests on signal-based strategies with vectorbt
- **[06-ml-prediction.ipynb](06-ml-prediction.ipynb)** — Deep dive into ML prediction models and walk-forward validation
- **[09-sentiment-analysis.ipynb](09-sentiment-analysis.ipynb)** — News sentiment pipeline and scoring details